# Lab 9 Eutectic growth

#### <p style="text-align: right;"> &#9989; **put your name here** </p>

### Due 11/24

<div align="left">
<img src="https://i.ibb.co/hRmPxmT1/Eutectic.jpg" width="500">
</div>

Image from *Materials science and engineering: an introduction / William D. Callister, Jr., David G. Rethwisch.–8th ed.*

---
In eutectic growth, the two solid phases separate as the solidification fronts advance. 
The governing equations of eutectic solidification using phase-field model are

$$\frac{\partial \phi}{\partial t} = -L \frac{\delta \mathcal{F}}{\delta \phi}=-L \bigg(\frac{\partial f}{\partial \phi} - \varepsilon_\phi^2 \nabla^2 \phi \bigg),$$

$$\frac{\partial C}{\partial t} = \nabla \cdot M_c \nabla \frac{\delta \mathcal{F} }{\delta C} = \nabla \cdot M_c \nabla \bigg(\frac{\partial f}{\partial C} - \varepsilon_\phi^2 \nabla^2 C \bigg),$$

$$\frac{\partial T}{\partial t} = \kappa \nabla^2 T + l\frac{\partial \phi}{\partial t},$$

where (an example free energy function) 

$$f(\phi,C,T) = w_\phi \phi^2 \big(1-\phi\big)^2 + \frac{1}{6}h(\phi) m(T) + h(\phi) w_s C^2 \big(1-C\big)^2 + \big(1-h(\phi)\big) w_l \big(0.5-C\big)^4.$$

### Visualize free energy function 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (needed for 3D)
%matplotlib ipympl

# --- Meshgrid equivalent ---
phi, C = np.meshgrid(np.arange(-0.1, 1.1 + 0.04, 0.04),
    np.arange(-0.1, 1.1 + 0.04, 0.04))

# --- Parameters ---
wp = 1
ws = 1
wl = 1
Tm = 1
T = 1.0

# --- Functions ---
fp = wp * phi**2 * (1 - phi)**2

h = phi**3 * (6*phi**2 - 15*phi + 10)
ft = h * np.arctan(T - Tm)

fs = ws * C**2 * (1 - C)**2 * h
fl = wl * (0.5 - C)**4 * (1 - h)

# --- Plot ---
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(
    phi, C, fl + fs + fp + ft,
    cmap='viridis', edgecolor='none'
)

ax.set_xlabel(r'$\phi$', fontsize=16)
ax.set_ylabel('C', fontsize=16)
ax.set_zlabel('f', fontsize=16)
ax.tick_params(labelsize=12)

plt.show()


---
## Simulation

The chemical potentials are

$$\frac{\partial f}{\partial \phi} = w_\phi 2 \phi \big(1-\phi \big) \big(1-2\phi\big) + \frac{1}{6} h'(\phi) m(T) + h'(\phi)w_s C^2 \big(1-C\big)^2 -h'(\phi)w_l \big(0.5-C\big)^4, $$

$$\frac{\partial f}{\partial C} = h(\phi) w_s 2 C \big(1-C\big) \big(1-2C\big) -\big(1-h(\phi)\big) w_l 4 \big(0.5-C\big)^3,$$

$$h(\phi) = \phi^3 \big( 6\phi^2 - 15\phi + 10\big),$$

$$h'(\phi) = 30\phi^2 \big(1-\phi\big)^2,$$

$$m(T) = \frac{\alpha}{\pi} \tan^{-1} \big[ \gamma \cdot (T-T_m) \big].$$

Here, we use a variable transport mobility of $C$, such that the diffusion is faster on the solid-liquid surface.:

$$M_c = 40 h'(\phi) + 0.05.$$

The simulation result will look like the video below.

In [ ]:
from IPython.display import YouTubeVideo

YouTubeVideo("StIJ6c2V-uA",width=640,height=360)

### simulation set up and initial conditions

Run the cell below to set up the simulation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time

# ----------------------------
# 2D square domain & grid
# ----------------------------
Ly = 3.0
Lx = 3.0

Ny = 100
Nx = 100

dx = Lx / Nx
dy = Ly / Ny
dt = 0.000125
Nstp = 5001  # 3001

# ----------------------------
# parameters
# ----------------------------
epsP = 0.012
M_phi = 500.0
L = 2.0
alph = 0.9
gamma = 10.0
Tm = 1.0
wp = 1.0 / 4.0

ws = 0.008
wl = 0.04

epsC = 0.0015

# ----------------------------
# fields (Ny, Nx)
# ----------------------------
phi  = np.zeros((Ny, Nx), dtype=float)
mu   = np.zeros((Ny, Nx), dtype=float)
LapP = np.zeros((Ny, Nx), dtype=float)

E    = np.zeros((Ny, Nx), dtype=float)
T    = -5.0 * np.ones((Ny, Nx), dtype=float)
LapT = np.zeros((Ny, Nx), dtype=float)

hf   = np.zeros((Ny, Nx), dtype=float)
dhf  = np.zeros((Ny, Nx), dtype=float)
muC  = np.zeros((Ny, Nx), dtype=float)
LapC = np.zeros((Ny, Nx), dtype=float)
M_C  = np.zeros((Ny, Nx), dtype=float)

# ----------------------------
# initial condition
# ----------------------------
for i in range(Ny):
    intf = np.random.randint(-1, 2) + 6  # 5, 6 or 7
    phi[i, :intf] = 1.0
    phi[i, intf:] = 0.0

Con = np.random.rand(Ny, Nx) * 0.01 + 0.495
tm = 0.0

import matplotlib.pyplot as plt

plt.ion()
fig, axes = plt.subplots(1, 3, figsize=(8, 3))  # 1 row, 3 columns
ax1, ax2, ax3 = axes

# --- phi ---
im1 = ax1.imshow(phi, origin='lower', vmin=-0.02, vmax=1.02, cmap='viridis')
ax1.set_title("phi")
fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

# --- T ---
im2 = ax2.imshow(T, origin='lower', cmap='hot')
ax2.set_title("T")
fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

# --- Con ---
im3 = ax3.imshow(Con, origin='lower', vmin=-0.02, vmax=1.02, cmap='jet')
ax3.set_title("Con")
fig.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------
# time iteration
# ----------------------------
for it in range(1, Nstp + 1):

    # convenience slices for interior
    i_s = slice(1, Ny - 1)
    j_s = slice(1, Nx - 1)

    # interpolation functions (C2 smoothed step)
    p = phi[i_s, j_s]
    hf[i_s, j_s]  = p**3 * (6.0 * p**2 - 15.0 * p + 10.0)
    dhf[i_s, j_s] = 30.0 * p**2 * (1.0 - p)**2

    # latent heat term E
    E[i_s, j_s] = 

    # chemical potential mu (order parameter part)
    mu[i_s, j_s] = 

    # Laplacian of phi (central differences)
    LapP[i_s, j_s] = 

    # total chemical potential mu
    mu[i_s, j_s] = mu[i_s, j_s] + (1.0 / 6.0)*E[i_s, j_s] - (epsP**2)*LapP[i_s, j_s]

    # concentration chemical potential muC (before diffusion term)
    muC[i_s, j_s] = 

    # Laplacian of Con (plain) for gradient energy
    LapC[i_s, j_s] = 

    # total chemical potential for C
    muC[i_s, j_s] = muC[i_s, j_s] - (epsC**2)*LapC[i_s, j_s]

    # boundary treatment for muC
    muC[ , ]   = 

    # variable mobility and divergence-form Laplacian of muC
    M_C[i_s, j_s] = 40.0 * dhf[i_s, j_s] + 0.05

    # flux divergence
    LapC[i_s, j_s] = 

    # temperature Laplacian
    LapT[i_s, j_s] = 

    # ----------------------------
    # updates
    # ----------------------------
    phi[i_s, j_s] = 
    Con[i_s, j_s] = 
    T[i_s, j_s]   = T[i_s, j_s]   + dt*LapT[i_s, j_s] - dt*L*M_phi*mu[i_s, j_s]

    # ----------------------------
    # boundary conditions (match MATLAB)
    # (Note: these are a mix of wrap/Neumann-like assignments as in your code)
    # ----------------------------
    phi[, ]   = 

    Con[, ]   = 

    T[, ]   = 

    # time
    tm += dt

    # ----------------------------
    # visualization every 20 iters
    # ----------------------------
    if it % 20 == 1:
        im1.set_data(phi)   
        im2.set_data(T)
        im3.set_data(Con)
        
        # Animaiton part (dosn't change)
        clear_output(wait=True) # Clear output for dynamic display
        display(fig)            # Reset display
        # fig.clear()             # Prevent overlapping and layered plots
        time.sleep(0.0002)         # Sleep for half a second to slow down the animation 


Do This - Describe the results you obtained. 

Do This - Based on your knowledge of materials science, how can you increase the lamina width? What parameters control the laminar width? 

### Great! You're done. Please upload your completed file to the drop box on the course webpage.